# PyTorch Cifar10 Image Classification

💡 This example demonstrates how to use *MLSteam Model SDK* (Archive `v2024.1`) to develop a Cifar10 model for image classification.

📘 Cifar10 was introduced by Krizhevsky et al. in [Learning multiple layers of features from tiny images](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf). The datasets are publicly available in the [author's site](https://www.cs.toronto.edu/~kriz/cifar.html) as well as through many popular ML framework libraries. The PyTorch model training code here originates from a [PyTorch tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html).

## Prepare Files

Download the `pytorch-cifar10-multiweights.zip` file and extract it.

The extracted files include:

```
pytorch-cifar10-multiweights/
  ├ dataset/
  │  └ samples/
  │    └ samples.pt  # a torch file containing 4 Cifar10 sample images
  ├ hooks/
  │  ├ load.py  # default loader hook 
  │  └ predict.py  # default predictor hook
  ├ model/
  │  ├ cifar_model-1.pth  # 1st state dict file
  │  └ cifar_model-2.pth  # 2nd state dict file
  ├ manifest-builtin-hooks.json  # manifest file using built-in hooks
  ├ manifest-custom-hooks.json  # manifest file using custom hooks
  ├ manifest.json  # manifest file using default hooks
  ├ my_hooks.py  # custom hooks
  ├ requirements.txt  # Python package dependencies
  └ train_multi_weights.py  # Cifar10 model definition & training script
```

Development typically starts with a model training script, which defines and trains the model from a training dataset. And they add manifest and optionally hook files for model packaging.

This example includes more files than necessary for demonstrating 3 ways to definining hooks. It also saves the model in 2 state dict files to show how the SDK deals with multi-weight-file models.

## Install Packages

1. Install `mlsteam-model-sdk` >= 0.7
1. Initilize SDK by `mlsteam-model-cli init -i`
1. Install themis by `mlsteam-model-cli install-themisdev`
1. Install cifar10 dependencies as follows.

In [1]:
# Install cifar10 dependencies
%cd pytorch-cifar10-multiweights
%pip install -r requirements.txt

/mlsteam/lab/pytorch-cifar10-multiweights


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


## Method 1: Default Hooks

This is the conventional way of defining hooks for model loading and prediction, which uses the `hooks/load.py` and `hooks/predict.py` files.

In [2]:
MODEL_NAME = 'torch-cifar10-demo'

In [3]:
# Create a model version under development
!mlsteam-model-cli dev mv add-local \
    --model-name "{MODEL_NAME}" \
    --version-name v1 \
    --hooks-dir hooks \
    --src-dir . \
    --submodel main model \
    --manifest manifest.json \
    -X .ipynb_checkpoints

2024-04-26 08:06:19 mlsteam-model-sdk [WARNING]: 'dev-c06cf915'
Generating dev mv record vuuid=dev-c06cf915...
Generating archive at /root/.mlsteam-model-sdk/models/download/dev-c06cf915.mlarchive...
Extracting archive at /root/.mlsteam-model-sdk/models/extract/dev-c06cf915...
Done.


In [4]:
# The new model version should be listed here
!mlsteam-model-cli mv list-local

total 1  dev 1

  muuid         model_name           vuuid       version_name     puuid     packaged   encrypted         download_time       
__local__   *torch-cifar10-demo   dev-c06cf915   *v1            __local__   True       False       2024-04-26T08:06:19.725841


In [5]:
# Load the model version
from mlsteam_model_sdk.sdk.model import Model
sdk_model = Model()
mv = sdk_model.load_model_version(model_name=MODEL_NAME, version_name='v1')

In [6]:
# Make prediction
import torch
ds = torch.load('dataset/samples/samples.pt')
print(mv.predict(ds['images']))
mv.close()

(tensor([2, 1, 8, 8]), ['bird', 'car', 'ship', 'ship'])


## Method 2: Custom Hooks

This method allows for specifying hook function defined in other places (E.g., `my_hooks.py` in this example), as long as the defining Python file is included in the source directory.

In [7]:
# Create a model version under development
!mlsteam-model-cli dev mv add-local \
    --model-name "{MODEL_NAME}" \
    --version-name v2 \
    --hooks-dir "" \
    --src-dir . \
    --submodel main model \
    --manifest manifest-custom-hooks.json \
    -X hooks \
    -X .ipynb_checkpoints

2024-04-26 08:06:22 mlsteam-model-sdk [WARNING]: 'dev-46674f24'
Generating dev mv record vuuid=dev-46674f24...
Generating archive at /root/.mlsteam-model-sdk/models/download/dev-46674f24.mlarchive...
Extracting archive at /root/.mlsteam-model-sdk/models/extract/dev-46674f24...
Done.


In [8]:
# Load the model version and make prediction
# NOTE: non-default hooks requires specifying `service_name` explicitly
with sdk_model.load_model_version(model_name=MODEL_NAME, version_name='v2') as mv:
    print(mv.predict(service_name='classify', inputs=ds['images']))

(tensor([2, 1, 8, 8]), ['bird', 'car', 'ship', 'ship'])


## Method 3: Built-in Hooks

This method allos for specifying hooks in manifest without writing hook files.

Below it loads the model and make prediction on CPU. To run on GPU, modify the `device` property to `cuda` within `manifest-builtin-hooks.json`, and transfer the input tensor to GPU.

In [9]:
# Create a model version under development
!mlsteam-model-cli dev mv add-local \
    --model-name "{MODEL_NAME}" \
    --version-name v3 \
    --hooks-dir "" \
    --src-dir . \
    --submodel main model \
    --manifest manifest-builtin-hooks.json \
    -X hooks \
    -X .ipynb_checkpoints

2024-04-26 08:06:22 mlsteam-model-sdk [WARNING]: 'dev-22667e89'
Generating dev mv record vuuid=dev-22667e89...
Generating archive at /root/.mlsteam-model-sdk/models/download/dev-22667e89.mlarchive...
Extracting archive at /root/.mlsteam-model-sdk/models/extract/dev-22667e89...
Done.


In [10]:
# Load the model version and make prediction
# NOTE: non-default hooks requires specifying `service_name` explicitly
with sdk_model.load_model_version(model_name=MODEL_NAME, version_name='v3') as mv:
    print(mv.predict(service_name='classify', inputs=ds['images']))

{'class_ids': tensor([2, 1, 8, 8]), 'probs': tensor([2.4466, 7.6294, 3.9962, 5.4334], grad_fn=<MaxBackward0>), 'class_names': ['bird', 'car', 'ship', 'ship']}


## Simulate Model Encryption 🔐

We can also simulate a model version packaged in encryption mode locally after everything works fine in plaintext mode.

In [11]:
# Convert a model version to simulated encryption mode
!mlsteam-model-cli dev mv update-local \
    --model-name "{MODEL_NAME}" \
    --version-name v3 \
    --encrypted \
    --sync

Updating dev mv record vuud=dev-22667e89...
Generating archive at /root/.mlsteam-model-sdk/models/download/dev-22667e89-enc.mlarchive...
Extracting archive at /root/.mlsteam-model-sdk/models/extract/dev-22667e89...
Done.


In [12]:
# List the model versions again
!mlsteam-model-cli mv list-local

total 3  dev 3

  muuid         model_name           vuuid       version_name     puuid     packaged   encrypted         download_time       
__local__   *torch-cifar10-demo   dev-c06cf915   *v1            __local__   True       False       2024-04-26T08:06:19.725841
__local__   *torch-cifar10-demo   dev-46674f24   *v2            __local__   True       False       2024-04-26T08:06:22.381246
__local__   *torch-cifar10-demo   dev-22667e89   *v3            __local__   True       True        2024-04-26T08:06:22.847044


In [13]:
# Load the model version and make prediction again
with sdk_model.load_model_version(model_name=MODEL_NAME, version_name='v3') as mv:
    print(mv.predict(service_name='classify', inputs=ds['images']))

{'class_ids': tensor([2, 1, 8, 8]), 'probs': tensor([2.4466, 7.6294, 3.9962, 5.4334], grad_fn=<MaxBackward0>), 'class_names': ['bird', 'car', 'ship', 'ship']}
